# 🚀 Day 2: Single Qubit Rotations in Qiskit

## 14-Day Quantum DevRel Bootcamp

**Goal:** Transition from manual NumPy to **Qiskit** — IBM's industry-standard quantum SDK.

**Today's Deliverables:**
1. ✅ Build circuits with Qiskit's `QuantumCircuit`
2. ✅ Understand rotation gates Rx(θ), Ry(θ), Rz(θ)
3. ✅ Visualize rotation trajectories on the Bloch sphere
4. ✅ Euler decomposition of arbitrary gates
5. ✅ Arbitrary state preparation with rotations

**Key transition:** Yesterday you implemented everything by hand in NumPy.  
Today you'll see that **Qiskit does the same math** — but connects to real hardware.

---

## Section 1: Your First Qiskit Circuit

In [1]:
# Core imports
import numpy as np
import plotly.graph_objects as go
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator

print("✅ Qiskit loaded successfully!")
print()

# ── Your first circuit: apply X gate ──
qc = QuantumCircuit(1)  # 1 qubit, starts in |0⟩
qc.x(0)                 # Apply Pauli-X to qubit 0

# Draw the circuit
print("Circuit diagram:")
print(qc.draw())
print()

# Extract the statevector
sv = Statevector.from_instruction(qc)
print(f"Statevector: {sv}")
print(f"As array:    {np.array(sv)}")
print()

# Compare to Day 1 manual calculation
X = np.array([[0, 1], [1, 0]], dtype=complex)
ket0 = np.array([1, 0], dtype=complex)
manual_result = X @ ket0
print(f"Day 1 NumPy:  {manual_result}")
print(f"Match: {np.allclose(np.array(sv), manual_result)} ✅")
print()
print("💡 Qiskit does the same linear algebra you did by hand!")

✅ Qiskit loaded successfully!

Circuit diagram:
   ┌───┐
q: ┤ X ├
   └───┘

Statevector: Statevector([0.+0.j, 1.+0.j],
            dims=(2,))
As array:    [0.+0.j 1.+0.j]

Day 1 NumPy:  [0.+0.j 1.+0.j]
Match: True ✅

💡 Qiskit does the same linear algebra you did by hand!


---
## Section 2: Common State Preparations

Qiskit always starts qubits in $|0\rangle$. To reach other states, we apply gates:

| Target State | Gates from $\vert 0\rangle$ | Circuit |
| :--- | :--- | :--- |
| $\vert 1\rangle$ | X | `qc.x(0)` |
| $\vert +\rangle$ | H | `qc.h(0)` |
| $\vert -\rangle$ | X then H | `qc.x(0); qc.h(0)` |
| $\vert +i\rangle$ | H then S | `qc.h(0); qc.s(0)` |
| $\vert -i\rangle$ | H then S† | `qc.h(0); qc.sdg(0)` |

In [2]:
# Prepare and verify all 6 cardinal states
print("📊 Cardinal Qubit States via Qiskit")
print("=" * 55)

states = {}

# |0⟩ — do nothing
qc = QuantumCircuit(1)
states['|0⟩'] = Statevector.from_instruction(qc)

# |1⟩ — apply X
qc = QuantumCircuit(1); qc.x(0)
states['|1⟩'] = Statevector.from_instruction(qc)

# |+⟩ — apply H
qc = QuantumCircuit(1); qc.h(0)
states['|+⟩'] = Statevector.from_instruction(qc)

# |-⟩ — apply X then H
qc = QuantumCircuit(1); qc.x(0); qc.h(0)
states['|-⟩'] = Statevector.from_instruction(qc)

# |+i⟩ — apply H then S
qc = QuantumCircuit(1); qc.h(0); qc.s(0)
states['|+i⟩'] = Statevector.from_instruction(qc)

# |-i⟩ — apply H then Sdg
qc = QuantumCircuit(1); qc.h(0); qc.sdg(0)
states['|-i⟩'] = Statevector.from_instruction(qc)

for name, sv in states.items():
    arr = np.array(sv)
    probs = np.abs(arr)**2
    print(f"{name:5s} = [{arr[0]:+.4f}, {arr[1]:+.4f}]  "
          f"P(0)={probs[0]:.2f}  P(1)={probs[1]:.2f}")

📊 Cardinal Qubit States via Qiskit
|0⟩   = [+1.0000+0.0000j, +0.0000+0.0000j]  P(0)=1.00  P(1)=0.00
|1⟩   = [+0.0000+0.0000j, +1.0000+0.0000j]  P(0)=0.00  P(1)=1.00
|+⟩   = [+0.7071+0.0000j, +0.7071+0.0000j]  P(0)=0.50  P(1)=0.50
|-⟩   = [+0.7071+0.0000j, -0.7071+0.0000j]  P(0)=0.50  P(1)=0.50
|+i⟩  = [+0.7071+0.0000j, +0.0000+0.7071j]  P(0)=0.50  P(1)=0.50
|-i⟩  = [+0.7071+0.0000j, +0.0000-0.7071j]  P(0)=0.50  P(1)=0.50


---
## Section 3: Rotation Gates — The Continuous Family

Pauli gates (X, Y, Z) are **discrete** — they're specific fixed operations.  
Rotation gates are their **continuous** generalization:

$$R_x(\theta) = e^{-i\theta X/2} = \cos(\theta/2)\,I - i\sin(\theta/2)\,X = \begin{pmatrix} \cos\theta/2 & -i\sin\theta/2 \\ -i\sin\theta/2 & \cos\theta/2 \end{pmatrix}$$

$$R_y(\theta) = e^{-i\theta Y/2} = \begin{pmatrix} \cos\theta/2 & -\sin\theta/2 \\ \sin\theta/2 & \cos\theta/2 \end{pmatrix}$$

$$R_z(\theta) = e^{-i\theta Z/2} = \begin{pmatrix} e^{-i\theta/2} & 0 \\ 0 & e^{i\theta/2} \end{pmatrix}$$

### Key relationships:
- $R_x(\pi) = -iX$ (X gate up to global phase)
- $R_y(\pi) = -iY$
- $R_z(\pi) = -iZ$

### Geometric meaning:
Each $R_n(\theta)$ rotates the Bloch vector by angle $\theta$ around the $n$-axis.

In [3]:
# Build rotation matrices manually and compare to Qiskit
print("🔄 Rotation Gates: Manual vs Qiskit")
print("=" * 55)

theta = np.pi / 3  # Test angle: 60°
c, s = np.cos(theta/2), np.sin(theta/2)

# ── Rx(θ) ──
manual_Rx = np.array([[c, -1j*s], [-1j*s, c]], dtype=complex)
qc = QuantumCircuit(1); qc.rx(theta, 0)
qiskit_Rx = Operator(qc).data

print(f"\nRx(π/3) manual:\n{manual_Rx}")
print(f"Rx(π/3) Qiskit:\n{qiskit_Rx}")
print(f"Match: {np.allclose(manual_Rx, qiskit_Rx)} ✅")

# ── Ry(θ) ──
manual_Ry = np.array([[c, -s], [s, c]], dtype=complex)
qc = QuantumCircuit(1); qc.ry(theta, 0)
qiskit_Ry = Operator(qc).data

print(f"\nRy(π/3) manual:\n{manual_Ry}")
print(f"Ry(π/3) Qiskit:\n{qiskit_Ry}")
print(f"Match: {np.allclose(manual_Ry, qiskit_Ry)} ✅")

# ── Rz(θ) ──
manual_Rz = np.array([[np.exp(-1j*theta/2), 0], [0, np.exp(1j*theta/2)]], dtype=complex)
qc = QuantumCircuit(1); qc.rz(theta, 0)
qiskit_Rz = Operator(qc).data

print(f"\nRz(π/3) manual:\n{manual_Rz}")
print(f"Rz(π/3) Qiskit:\n{qiskit_Rz}")
print(f"Match: {np.allclose(manual_Rz, qiskit_Rz)} ✅")

# ── Verify: Rx(π) ≡ X up to global phase ──
print("\n" + "=" * 55)
print("🔗 Pauli gates are π-rotations!")

qc_rx_pi = QuantumCircuit(1); qc_rx_pi.rx(np.pi, 0)
sv_rx = Statevector.from_instruction(qc_rx_pi)

qc_x = QuantumCircuit(1); qc_x.x(0)
sv_x = Statevector.from_instruction(qc_x)

print(f"Rx(π)|0⟩ = {np.array(sv_rx)}")
print(f"X|0⟩     = {np.array(sv_x)}")
print(f"Equivalent (up to global phase): {sv_rx.equiv(sv_x)} ✅")

🔄 Rotation Gates: Manual vs Qiskit

Rx(π/3) manual:
[[ 0.8660254+0.j  -0.       -0.5j]
 [-0.       -0.5j  0.8660254+0.j ]]
Rx(π/3) Qiskit:
[[0.8660254+0.j  0.       -0.5j]
 [0.       -0.5j 0.8660254+0.j ]]
Match: True ✅

Ry(π/3) manual:
[[ 0.8660254+0.j -0.5      +0.j]
 [ 0.5      +0.j  0.8660254+0.j]]
Ry(π/3) Qiskit:
[[ 0.8660254+0.j -0.5      +0.j]
 [ 0.5      +0.j  0.8660254+0.j]]
Match: True ✅

Rz(π/3) manual:
[[0.8660254-0.5j 0.       +0.j ]
 [0.       +0.j  0.8660254+0.5j]]
Rz(π/3) Qiskit:
[[0.8660254-0.5j 0.       +0.j ]
 [0.       +0.j  0.8660254+0.5j]]
Match: True ✅

🔗 Pauli gates are π-rotations!
Rx(π)|0⟩ = [6.123234e-17+0.j 0.000000e+00-1.j]
X|0⟩     = [0.+0.j 1.+0.j]
Equivalent (up to global phase): True ✅


---
## Section 4: Bloch Sphere Trajectories

The most intuitive way to understand rotations: **watch them move on the Bloch sphere**.

- **Rx(θ)** rotates around the X-axis → traces a circle in the YZ plane
- **Ry(θ)** rotates around the Y-axis → traces a circle in the XZ plane  
- **Rz(θ)** rotates around the Z-axis → traces a circle in the XY plane

### Helper functions for Bloch coordinates

In [4]:
# Pauli matrices for Bloch coordinate computation
PAULI_X = np.array([[0, 1], [1, 0]], dtype=complex)
PAULI_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
PAULI_Z = np.array([[1, 0], [0, -1]], dtype=complex)

def state_to_bloch(state) -> tuple:
    """Convert a Statevector or numpy array to Bloch coordinates."""
    if isinstance(state, Statevector):
        s = np.array(state)
    else:
        s = np.array(state, dtype=complex)
    x = np.real(np.conj(s) @ PAULI_X @ s)
    y = np.real(np.conj(s) @ PAULI_Y @ s)
    z = np.real(np.conj(s) @ PAULI_Z @ s)
    return float(x), float(y), float(z)


def make_sphere_wireframe():
    """Create plotly traces for a translucent Bloch sphere."""
    traces = []
    
    # Sphere surface
    u = np.linspace(0, 2*np.pi, 30)
    v = np.linspace(0, np.pi, 30)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones(len(u)), np.cos(v))
    
    traces.append(go.Surface(
        x=xs, y=ys, z=zs,
        opacity=0.1,
        colorscale=[[0, 'lightgray'], [1, 'lightgray']],
        showscale=False, hoverinfo='skip'
    ))
    
    # Coordinate axes
    for coords, color, label in [
        ([1.4,0,0], 'red', 'X'), ([0,1.4,0], 'green', 'Y'), ([0,0,1.4], 'blue', 'Z|0⟩')
    ]:
        traces.append(go.Scatter3d(
            x=[0, coords[0]], y=[0, coords[1]], z=[0, coords[2]],
            mode='lines+text', text=['', label],
            textposition='top center',
            line=dict(color=color, width=3),
            showlegend=False, hoverinfo='skip'
        ))
    
    return traces


def make_bloch_layout(title='Bloch Sphere'):
    """Standard layout for Bloch sphere plots."""
    return go.Layout(
        title=title,
        scene=dict(
            xaxis=dict(range=[-1.5, 1.5], title='X'),
            yaxis=dict(range=[-1.5, 1.5], title='Y'),
            zaxis=dict(range=[-1.5, 1.5], title='Z'),
            aspectmode='cube'
        ),
        width=750, height=700
    )


print("✅ Bloch sphere helper functions defined.")

✅ Bloch sphere helper functions defined.


### 4a. Rx Trajectory: Rotation around the X-axis

Starting from $|0\rangle$ (north pole), $R_x(\theta)$ sweeps through:  
$|0\rangle \to |{-i}\rangle \to |1\rangle \to |{+i}\rangle \to |0\rangle$  
(a great circle in the **YZ plane**)

In [5]:
# Rx trajectory: sweep θ from 0 to 2π
n_points = 60
angles = np.linspace(0, 2*np.pi, n_points)
coords_rx = []

for theta in angles:
    qc = QuantumCircuit(1)
    qc.rx(theta, 0)
    sv = Statevector.from_instruction(qc)
    coords_rx.append(state_to_bloch(sv))

xs, ys, zs = zip(*coords_rx)

fig = go.Figure(data=make_sphere_wireframe())

# Trajectory line
fig.add_trace(go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode='lines+markers',
    marker=dict(size=3, color=np.degrees(angles), colorscale='Reds', colorbar=dict(title='θ (deg)')),
    line=dict(color='red', width=4),
    name='Rx(θ)|0⟩',
    hovertemplate='θ=%{customdata:.0f}°<br>x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}',
    customdata=np.degrees(angles)
))

# Mark key states
key_points = [(0, '|0⟩'), (np.pi/2, 'Rx(π/2)|0⟩'), (np.pi, '|1⟩'), (3*np.pi/2, 'Rx(3π/2)|0⟩')]
for angle, label in key_points:
    qc = QuantumCircuit(1); qc.rx(angle, 0)
    bx, by, bz = state_to_bloch(Statevector.from_instruction(qc))
    fig.add_trace(go.Scatter3d(
        x=[bx], y=[by], z=[bz],
        mode='markers+text', text=[label], textposition='top center',
        marker=dict(size=10, color='darkred', symbol='diamond'),
        showlegend=False
    ))

fig.update_layout(make_bloch_layout('Rx(θ)|0⟩ Trajectory — Rotation Around X-Axis'))
fig.show()

### 4b. Ry Trajectory: Rotation around the Y-axis

Starting from $|0\rangle$, $R_y(\theta)$ sweeps through:  
$|0\rangle \to |+\rangle \to |1\rangle \to |-\rangle \to |0\rangle$  
(a great circle in the **XZ plane**)

💡 Ry is special: it keeps all amplitudes **real**. Perfect for state preparation!

In [6]:
# Ry trajectory
coords_ry = []
for theta in angles:
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    sv = Statevector.from_instruction(qc)
    coords_ry.append(state_to_bloch(sv))

xs, ys, zs = zip(*coords_ry)

fig = go.Figure(data=make_sphere_wireframe())

fig.add_trace(go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode='lines+markers',
    marker=dict(size=3, color=np.degrees(angles), colorscale='Greens', colorbar=dict(title='θ (deg)')),
    line=dict(color='green', width=4),
    name='Ry(θ)|0⟩',
    hovertemplate='θ=%{customdata:.0f}°<br>x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}',
    customdata=np.degrees(angles)
))

key_points = [(0, '|0⟩'), (np.pi/2, '|+⟩'), (np.pi, '|1⟩'), (3*np.pi/2, '|-⟩')]
for angle, label in key_points:
    qc = QuantumCircuit(1); qc.ry(angle, 0)
    bx, by, bz = state_to_bloch(Statevector.from_instruction(qc))
    fig.add_trace(go.Scatter3d(
        x=[bx], y=[by], z=[bz],
        mode='markers+text', text=[label], textposition='top center',
        marker=dict(size=10, color='darkgreen', symbol='diamond'),
        showlegend=False
    ))

fig.update_layout(make_bloch_layout('Ry(θ)|0⟩ Trajectory — Rotation Around Y-Axis'))
fig.show()

### 4c. Rz Trajectory: Rotation around the Z-axis

Starting from $|+\rangle$ (equator), $R_z(\theta)$ sweeps around the equator:  
$|+\rangle \to |{+i}\rangle \to |-\rangle \to |{-i}\rangle \to |+\rangle$  

⚠️ Note: If we start from $|0\rangle$, Rz only adds a global phase — the Bloch vector **doesn't move**!  
That's because $|0\rangle$ sits ON the Z-axis. So we start from $|+\rangle$ instead.

In [7]:
# Rz trajectory — starting from |+⟩ (equator)
coords_rz = []
for theta in angles:
    qc = QuantumCircuit(1)
    qc.h(0)           # Start from |+⟩
    qc.rz(theta, 0)   # Rotate around Z-axis
    sv = Statevector.from_instruction(qc)
    coords_rz.append(state_to_bloch(sv))

xs, ys, zs = zip(*coords_rz)

fig = go.Figure(data=make_sphere_wireframe())

fig.add_trace(go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode='lines+markers',
    marker=dict(size=3, color=np.degrees(angles), colorscale='Blues', colorbar=dict(title='θ (deg)')),
    line=dict(color='blue', width=4),
    name='Rz(θ)|+⟩',
    hovertemplate='θ=%{customdata:.0f}°<br>x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}',
    customdata=np.degrees(angles)
))

key_points = [(0, '|+⟩'), (np.pi/2, '|+i⟩'), (np.pi, '|-⟩'), (3*np.pi/2, '|-i⟩')]
for angle, label in key_points:
    qc = QuantumCircuit(1); qc.h(0); qc.rz(angle, 0)
    bx, by, bz = state_to_bloch(Statevector.from_instruction(qc))
    fig.add_trace(go.Scatter3d(
        x=[bx], y=[by], z=[bz],
        mode='markers+text', text=[label], textposition='top center',
        marker=dict(size=10, color='darkblue', symbol='diamond'),
        showlegend=False
    ))

fig.update_layout(make_bloch_layout('Rz(θ)|+⟩ Trajectory — Rotation Around Z-Axis'))
fig.show()

### 4d. All Three Trajectories Together

See all three rotation axes on one sphere for comparison.

In [8]:
# Combined plot: all three rotation trajectories
fig = go.Figure(data=make_sphere_wireframe())

for coords, color, name in [
    (coords_rx, 'red', 'Rx(θ)|0⟩ (YZ plane)'),
    (coords_ry, 'green', 'Ry(θ)|0⟩ (XZ plane)'),
    (coords_rz, 'blue', 'Rz(θ)|+⟩ (XY plane)')
]:
    xs, ys, zs = zip(*coords)
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode='lines',
        line=dict(color=color, width=5),
        name=name
    ))

fig.update_layout(make_bloch_layout('All Rotation Trajectories'))
fig.show()

---
## Section 5: Euler Decomposition (ZYZ)

### The Most Important Theorem for Quantum Hardware

**Any** single-qubit unitary $U$ can be decomposed as:

$$U = e^{i\alpha} \, R_z(\beta) \, R_y(\gamma) \, R_z(\delta)$$

This means:
1. Only **two types** of rotation (Ry + Rz) can produce **any** single-qubit gate
2. Hardware only needs to physically implement Ry and Rz
3. On IBM hardware, Rz is a **virtual gate** (zero error!)

### Let's verify: Decompose Hadamard into rotations

$$H = R_y(\pi/2) \cdot R_z(\pi) \quad \text{(up to global phase)}$$

In [9]:
print("🔧 Euler Decomposition: Breaking Gates into Rotations")
print("=" * 55)

# Hadamard = Ry(π/2) · Rz(π)  (matrix multiplication order)
# In Qiskit circuit order: rz(π) first, then ry(π/2)
qc_h_rotation = QuantumCircuit(1)
qc_h_rotation.rz(np.pi, 0)       # First gate applied
qc_h_rotation.ry(np.pi / 2, 0)   # Second gate applied

qc_h_native = QuantumCircuit(1)
qc_h_native.h(0)

# Compare: Do they produce the same state from |0⟩?
sv_rotation = Statevector.from_instruction(qc_h_rotation)
sv_native = Statevector.from_instruction(qc_h_native)

print("Hadamard via rotations: Rz(π) → Ry(π/2)")
print(f"  Rotation result: {np.array(sv_rotation)}")
print(f"  Native H result: {np.array(sv_native)}")
print(f"  Equivalent: {sv_rotation.equiv(sv_native)} ✅")
print()

# Compare the full unitary matrices
op_rotation = Operator(qc_h_rotation)
op_native = Operator(qc_h_native)
print("Matrix comparison:")
print(f"  Rotation:\n  {op_rotation.data}")
print(f"  Native H:\n  {op_native.data}")
print(f"  Equal up to global phase: {op_rotation.equiv(op_native)} ✅")
print()

# ── More decompositions ──
print("More gate decompositions:")
print("-" * 55)

# S gate = Rz(π/2) up to global phase
qc_s = QuantumCircuit(1); qc_s.s(0)
qc_rz_s = QuantumCircuit(1); qc_rz_s.rz(np.pi/2, 0)
print(f"S gate ≡ Rz(π/2): {Operator(qc_s).equiv(Operator(qc_rz_s))} ✅")

# T gate = Rz(π/4) up to global phase
qc_t = QuantumCircuit(1); qc_t.t(0)
qc_rz_t = QuantumCircuit(1); qc_rz_t.rz(np.pi/4, 0)
print(f"T gate ≡ Rz(π/4): {Operator(qc_t).equiv(Operator(qc_rz_t))} ✅")

# X gate = Rx(π) up to global phase
qc_x = QuantumCircuit(1); qc_x.x(0)
qc_rx_pi = QuantumCircuit(1); qc_rx_pi.rx(np.pi, 0)
print(f"X gate ≡ Rx(π):   {Operator(qc_x).equiv(Operator(qc_rx_pi))} ✅")
print()

print("💡 KEY INSIGHT: Named gates (H, S, T, X) are just")
print("   specific rotation angles! The rotation gates are more fundamental.")

🔧 Euler Decomposition: Breaking Gates into Rotations
Hadamard via rotations: Rz(π) → Ry(π/2)
  Rotation result: [4.32978028e-17-0.70710678j 4.32978028e-17-0.70710678j]
  Native H result: [0.70710678+0.j 0.70710678+0.j]
  Equivalent: True ✅

Matrix comparison:
  Rotation:
  [[ 4.32978028e-17-0.70710678j -4.32978028e-17-0.70710678j]
 [ 4.32978028e-17-0.70710678j  4.32978028e-17+0.70710678j]]
  Native H:
  [[ 0.70710678+0.j  0.70710678+0.j]
 [ 0.70710678+0.j -0.70710678+0.j]]
  Equal up to global phase: True ✅

More gate decompositions:
-------------------------------------------------------
S gate ≡ Rz(π/2): True ✅
T gate ≡ Rz(π/4): True ✅
X gate ≡ Rx(π):   True ✅

💡 KEY INSIGHT: Named gates (H, S, T, X) are just
   specific rotation angles! The rotation gates are more fundamental.


---
## Section 6: Arbitrary State Preparation

Any single-qubit state can be parametrized on the Bloch sphere:

$$|\psi\rangle = \cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$$

To prepare this from $|0\rangle$:
1. $R_y(\theta)$: Sets the **polar angle** (latitude) — how much $|0\rangle$ vs $|1\rangle$
2. $R_z(\phi)$: Sets the **azimuthal phase** (longitude) — the relative phase

$$|\psi\rangle = R_z(\phi) \, R_y(\theta) |0\rangle \quad \text{(up to global phase)}$$

In [10]:
def prepare_state(theta: float, phi: float) -> Statevector:
    """Prepare |ψ⟩ = cos(θ/2)|0⟩ + e^(iφ)sin(θ/2)|1⟩."""
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    qc.rz(phi, 0)
    return Statevector.from_instruction(qc)


# Prepare a grid of states covering the Bloch sphere
print("🌐 Preparing states across the Bloch sphere")
print("=" * 55)

fig = go.Figure(data=make_sphere_wireframe())

# Grid of states
theta_vals = np.linspace(0, np.pi, 7)      # 0 to π (north to south)
phi_vals = np.linspace(0, 2*np.pi, 13)[:-1] # 0 to 2π (around equator)

all_x, all_y, all_z, all_colors, all_text = [], [], [], [], []

for theta in theta_vals:
    for phi in phi_vals:
        sv = prepare_state(theta, phi)
        bx, by, bz = state_to_bloch(sv)
        all_x.append(bx)
        all_y.append(by)
        all_z.append(bz)
        all_colors.append(np.degrees(theta))
        all_text.append(f'θ={np.degrees(theta):.0f}° φ={np.degrees(phi):.0f}°')

fig.add_trace(go.Scatter3d(
    x=all_x, y=all_y, z=all_z,
    mode='markers',
    marker=dict(
        size=5,
        color=all_colors,
        colorscale='Viridis',
        colorbar=dict(title='θ (deg)'),
        opacity=0.8
    ),
    text=all_text,
    hovertemplate='%{text}<br>x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}',
    name='Prepared states'
))

# Mark the 6 cardinal states
cardinal = [
    (0, 0, '|0⟩'), (np.pi, 0, '|1⟩'),
    (np.pi/2, 0, '|+⟩'), (np.pi/2, np.pi, '|-⟩'),
    (np.pi/2, np.pi/2, '|+i⟩'), (np.pi/2, 3*np.pi/2, '|-i⟩')
]
for theta, phi, label in cardinal:
    sv = prepare_state(theta, phi)
    bx, by, bz = state_to_bloch(sv)
    fig.add_trace(go.Scatter3d(
        x=[bx], y=[by], z=[bz],
        mode='markers+text', text=[label], textposition='top center',
        marker=dict(size=10, color='red', symbol='diamond'),
        showlegend=False
    ))

fig.update_layout(make_bloch_layout('Arbitrary State Preparation: Ry(θ)·Rz(φ)|0⟩'))
fig.show()

print(f"\nPrepared {len(all_x)} states covering the Bloch sphere.")
print("Each state was created with just Ry(θ) + Rz(φ) from |0⟩.")

🌐 Preparing states across the Bloch sphere



Prepared 84 states covering the Bloch sphere.
Each state was created with just Ry(θ) + Rz(φ) from |0⟩.


---
## Section 7: Circuit Visualization & Transpilation

Qiskit can draw circuits in multiple formats and **transpile** them to native hardware gate sets.

IBM's native gates:
- `rz(θ)` — virtual gate (zero error!)
- `sx` — $\sqrt{X}$ gate ($R_x(\pi/2)$)
- `x` — Pauli-X gate
- `cx` — CNOT (two-qubit, for Day 3)

In [11]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

print("📐 Circuit Diagrams & Transpilation")
print("=" * 55)

# Build a multi-gate circuit
qc = QuantumCircuit(1)
qc.h(0)
qc.t(0)
qc.h(0)

print("Original circuit (H → T → H):")
print(qc.draw())
print()

# Transpile to IBM basis gates
pm = generate_preset_pass_manager(
    optimization_level=2,
    basis_gates=['rz', 'sx', 'x', 'cx']
)
transpiled = pm.run(qc)

print("Transpiled to IBM native gates (rz, sx, x):")
print(transpiled.draw())
print()

# Count gates
print("Gate counts:")
ops = transpiled.count_ops()
for gate, count in ops.items():
    print(f"  {gate}: {count}")
print()

# Verify: same unitary
print(f"Same unitary? {Operator(qc).equiv(Operator(transpiled))} ✅")
print()
print("💡 The transpiler decomposed H and T into rotations.")
print("   On real hardware, Rz is FREE (virtual gate).")
print("   Only sx/x gates cost physical microwave pulses.")

📐 Circuit Diagrams & Transpilation
Original circuit (H → T → H):
   ┌───┐┌───┐┌───┐
q: ┤ H ├┤ T ├┤ H ├
   └───┘└───┘└───┘

Transpiled to IBM native gates (rz, sx, x):
global phase: 13π/8
   ┌─────────┐┌────┐┌───────────┐┌────┐┌─────────┐
q: ┤ Rz(π/2) ├┤ √X ├┤ Rz(-3π/4) ├┤ √X ├┤ Rz(π/2) ├
   └─────────┘└────┘└───────────┘└────┘└─────────┘

Gate counts:
  rz: 3
  sx: 2

Same unitary? True ✅

💡 The transpiler decomposed H and T into rotations.
   On real hardware, Rz is FREE (virtual gate).
   Only sx/x gates cost physical microwave pulses.


---
## Section 8: Gate Sequence Composition

Understanding how gates **compose** is critical for circuit design.

Key concept: In a circuit `qc.a(0); qc.b(0)`, the **matrix** is $B \cdot A$.  
Gates apply left-to-right in code, but **right-to-left** in matrix multiplication.

Let's verify this with a concrete example.

In [12]:
print("🔗 Gate Composition: Circuit Order vs Matrix Order")
print("=" * 55)

# Circuit: X then H
qc = QuantumCircuit(1)
qc.x(0)   # Applied first
qc.h(0)   # Applied second

print("Circuit: X → H")
print(qc.draw())
print()

# Matrix: H · X  (reverse order!)
X = Operator.from_label('X').data
H = Operator.from_label('H').data
manual_matrix = H @ X

qiskit_matrix = Operator(qc).data

print(f"Manual (H @ X):\n{manual_matrix}")
print(f"\nQiskit result:\n{qiskit_matrix}")
print(f"\nMatch: {np.allclose(manual_matrix, qiskit_matrix)} ✅")
print()

# Demonstrate: X|0⟩ = |1⟩, then H|1⟩ = |-⟩
sv = Statevector.from_instruction(qc)
print(f"Result: X|0⟩ → |1⟩ → H|1⟩ = |-⟩")
print(f"State: {np.array(sv)}")
expected_minus = Statevector([1/np.sqrt(2), -1/np.sqrt(2)])
print(f"Is |-⟩? {sv.equiv(expected_minus)} ✅")
print()
print("💡 REMEMBER: Code order (left→right) = REVERSE matrix order")
print("   qc.a(); qc.b()  →  Matrix = B · A  →  State = B(A|0⟩)")

🔗 Gate Composition: Circuit Order vs Matrix Order
Circuit: X → H
   ┌───┐┌───┐
q: ┤ X ├┤ H ├
   └───┘└───┘

Manual (H @ X):
[[ 0.70710678+0.j  0.70710678+0.j]
 [-0.70710678+0.j  0.70710678+0.j]]

Qiskit result:
[[ 0.70710678+0.j  0.70710678+0.j]
 [-0.70710678+0.j  0.70710678+0.j]]

Match: True ✅

Result: X|0⟩ → |1⟩ → H|1⟩ = |-⟩
State: [ 0.70710678+0.j -0.70710678+0.j]
Is |-⟩? True ✅

💡 REMEMBER: Code order (left→right) = REVERSE matrix order
   qc.a(); qc.b()  →  Matrix = B · A  →  State = B(A|0⟩)


---
## 🎯 Day 2 Summary & Deliverables

### What you learned today:
1. **Qiskit Basics** — `QuantumCircuit`, `Statevector`, `Operator`
2. **Rotation Gates** — $R_x(\theta)$, $R_y(\theta)$, $R_z(\theta)$ as continuous gate families
3. **Bloch Trajectories** — Each rotation traces a great circle around its axis
4. **Euler Decomposition** — Any single-qubit gate = $R_z \cdot R_y \cdot R_z$
5. **State Preparation** — $R_y(\theta) \cdot R_z(\phi)$ reaches any point on the Bloch sphere
6. **Transpilation** — How Qiskit compiles to native hardware gates
7. **Composition** — Circuit order vs matrix multiplication order

### Your deliverables:
- ✅ Rotation matrix implementations (manual vs Qiskit)
- ✅ Interactive Bloch sphere trajectories
- ✅ Hadamard decomposed into rotations
- ✅ Arbitrary state preparation circuit
- ✅ Portfolio artifact: "The Geometry of Single-Qubit Gates"

### Key takeaway for interviews:
> "Every single-qubit gate decomposes into Rz-Ry-Rz rotations. On IBM hardware, Rz is a virtual gate with zero error — so the real cost comes from two-qubit gates like CNOT, not single-qubit operations."

---
**Tomorrow (Day 3):** Multi-Qubit Systems & Entanglement — CNOT, Bell states, and the power of quantum correlations.